# Day 22: Window Functions - Rolling, Shift, Diff

## Shift
Window functions look at a value in relation to its neighboring rows rather than in isolation — a foundational idea for any time-ordered or sequential data. `shift()` moves a column's values down (or up) by a given number of rows without reordering anything, which lets you line up "today's value" next to "yesterday's value" in the same row for easy comparison. This is the building block behind computing day-over-day change, lag features for forecasting models, and comparisons like "did sales go up or down since last time." Below, `shift(1)` creates a new column holding each row's *previous* row's Sales value — notice the first row becomes `NaN` since there's no row before it to pull from.</cell id="cell-1">


In [ ]:
import pandas as pd

df = pd.DataFrame({'Day': range(1, 8), 'Sales': [10, 12, 9, 15, 20, 18, 25]})
df['Prev_Day_Sales'] = df['Sales'].shift(1)  # each row now also shows the previous row's Sales value
print(df)

## Diff & Percent Change
Instead of manually shifting and subtracting, pandas gives you `diff()` and `pct_change()` as direct shortcuts for the two most common row-to-row comparisons: the raw difference and the percentage change. `diff()` computes `current - previous` for each row, useful for tracking absolute change (e.g. sales went up by 3 units). `pct_change()` computes `(current - previous) / previous`, which is more useful when you care about relative growth (e.g. sales grew by 20%) rather than the raw number, especially when comparing across series of very different scales. As with `shift`, the first row in both new columns will be `NaN` since there's nothing before it to compare against — the rest show the change from each row to the one before it.</cell id="cell-3">


In [ ]:
df['Change'] = df['Sales'].diff()  # absolute change from the previous row (current - previous)
df['Pct_Change'] = df['Sales'].pct_change().round(2)  # relative change as a fraction, e.g. 0.2 = 20% growth
print(df)

## Rolling Window
As covered with time series, a rolling window computes a statistic over a fixed-size sliding window of consecutive rows — here revisited in the context of general window functions rather than dates specifically. It's the standard way to smooth out noisy data or generate features like "average of the last N periods," which is common both in reporting (smoothed trend lines) and in feature engineering for forecasting models. `window=3` means each value is the average of that row plus the 2 rows before it, so the window "slides" one row at a time down the column. In the output, the first 2 rows of `3Day_Avg` are `NaN` (not enough prior rows yet to fill a 3-row window), and from row 3 onward each value is the mean of the current and 2 preceding Sales values.</cell id="cell-5">


In [ ]:
# average of each row plus the 2 rows before it (a sliding 3-row window)
df['3Day_Avg'] = df['Sales'].rolling(window=3).mean()
print(df)

## Expanding & Cumulative
Unlike a rolling window (fixed size that slides), an expanding window keeps growing to include every row from the start up through the current one — so it answers "what's the total/max/etc. so far" rather than "what's it like recently." `cumsum()` is a specific, commonly used cumulative calculation that gives the running total up to each row, useful for things like year-to-date totals. `expanding().max()` is the more general form, letting you apply any aggregation (here, max) to that ever-growing window, giving a "running maximum" — the highest value seen up to and including that row. In the output, `Cumulative_Sales` should steadily increase (since Sales are positive), and `Running_Max` should be non-decreasing, only changing when a new all-time-high Sales value appears.</cell id="cell-7">


In [ ]:
df['Cumulative_Sales'] = df['Sales'].cumsum()  # running total of Sales up to and including each row
df['Running_Max'] = df['Sales'].expanding().max()  # highest Sales value seen so far, up to each row
print(df)